# M³ fast: PANGAEA descriptive metadata, Mistral Small

This notebook runs the **two-agent, one-revision-round** M³ path with SURF `mistralai/Mistral-Small-3.2-24B-Instruct-2506` on every PANGAEA CSV.

The standard is `croissant_pangaea_standard`: **name, description, and keywords** only. It does not extract spatial or temporal coverage, and it does not run iNaturalist.

Results are written to `outputs/m3_fast_pangaea_descriptive_Mistral-Small-3.2-24B-Instruct-2506`. Other M³ fast folders are not read or written. Rerunning skips successes already stored here. Failures stay retryable. Set `M3_MAX_NEW_RUNS=0` to refresh the manifest without new model calls.

In [1]:
import hashlib
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from tqdm.auto import tqdm

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'config.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the metadata_agent checkout')
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

from src.orchestrator.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS, get_schema_for_standard

MODEL = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'
TOPOLOGY = 'fast'
MAX_NEW_RUNS = int(os.getenv('M3_MAX_NEW_RUNS', '-1'))  # -1 runs all pending files
STANDARD = 'croissant_pangaea_standard'
FIELDS = ('name', 'description', 'keywords')
DATA_DIR = ROOT / 'data/experiment/pangaea_datasets'
OUT = ROOT / 'outputs/m3_fast_pangaea_descriptive_Mistral-Small-3.2-24B-Instruct-2506'
OUT.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)
if MODEL != 'mistralai/Mistral-Small-3.2-24B-Instruct-2506' or TOPOLOGY != 'fast':
    raise ValueError('This notebook is pinned to Mistral Small and the fast topology')
assert set(get_schema_for_standard(STANDARD).model_fields) == set(FIELDS)
print('Project:', ROOT)
print('Model:', MODEL)
print('Standard:', STANDARD)
print('Output:', OUT)
print('New M³ runs allowed this session:', 'all pending' if MAX_NEW_RUNS < 0 else MAX_NEW_RUNS)
print('SURF credentials present:', bool(os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')))


Project: /home/com3dian/Github/metadata_agent
Model: mistralai/Mistral-Small-3.2-24B-Instruct-2506
Standard: croissant_pangaea_standard
Output: /home/com3dian/Github/metadata_agent/outputs/m3_fast_pangaea_descriptive_Mistral-Small-3.2-24B-Instruct-2506
New M³ runs allowed this session: all pending
SURF credentials present: True


In [2]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temporary.replace(path)


def m3_result_path(stem: str) -> Path:
    return OUT / 'm3' / 'pangaea' / f'{stem}.json'


def valid_m3_success(path: Path, item: dict) -> bool:
    if not path.is_file():
        return False
    try:
        record = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False
    return (record.get('status') == 'ok'
            and record.get('model') == MODEL
            and record.get('topology') == TOPOLOGY
            and record.get('standard') == STANDARD
            and record.get('source_sha256') == item['source_sha256']
            and isinstance(record.get('metadata'), dict)
            and set(FIELDS).issubset(record['metadata']))


selected = []
for source in sorted(DATA_DIR.glob('*.csv')):
    selected.append({'id': source.stem, 'source_sha256': sha256_file(source)})
if not selected:
    raise FileNotFoundError(f'No CSV files in {DATA_DIR}')
manifest = {'model': MODEL, 'topology': TOPOLOGY, 'standard': STANDARD, 'datasets': selected}
write_json_atomic(OUT / 'manifest.json', manifest)
ready = sum(valid_m3_success(m3_result_path(item['id']), item) for item in selected)
print('pangaea', len(selected), 'datasets;', ready, 'successes already saved')
print('Manifest:', OUT / 'manifest.json')


pangaea 75 datasets; 0 successes already saved
Manifest: /home/com3dian/Github/metadata_agent/outputs/m3_fast_pangaea_descriptive_Mistral-Small-3.2-24B-Instruct-2506/manifest.json


In [3]:
import logging

log = logging.getLogger()
log.setLevel(logging.INFO)
for handler in log.handlers[:]:
    log.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
log.addHandler(handler)


def run_m3_one(item: dict) -> dict:
    source = DATA_DIR / f"{item['id']}.csv"
    base = {'dataset_id': item['id'], 'csv': str(source), 'source_sha256': item['source_sha256'],
            'method': 'M3_fast', 'provider': 'surf', 'model': MODEL,
            'topology': TOPOLOGY, 'standard': STANDARD,
            'finished_utc': datetime.now(timezone.utc).isoformat()}
    started = time.monotonic()
    orchestrator = None
    try:
        orchestrator = Orchestrator(topology_name=TOPOLOGY, model_name=MODEL, provider='surf')
        result = orchestrator.run(
            source=str(source), metadata_standard=METADATA_STANDARDS[STANDARD],
            metadata_standard_name=STANDARD, name=item['id'],
        )
        metadata = result.final_metadata if result is not None else None
        if result is None or not result.success:
            raise RuntimeError((result.error if result is not None else 'Planning failed') or 'Run failed')
        if not isinstance(metadata, dict) or not set(FIELDS).issubset(metadata):
            raise ValueError(f'Missing requested metadata fields: {sorted(set(FIELDS)-set(metadata or {}))}')
        metadata = get_schema_for_standard(STANDARD).model_validate(metadata).model_dump(
            mode='json', by_alias=True
        )
        return {**base, 'status': 'ok', 'metadata': metadata,
                'plan_steps_count': result.plan_steps_count,
                'steps_completed': result.steps_completed,
                'trace_path': result.trace_path,
                'runtime_seconds': round(time.monotonic()-started, 2)}
    except Exception as exc:
        return {**base, 'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
                'trace_path': getattr(orchestrator, 'last_trace_path', None),
                'runtime_seconds': round(time.monotonic()-started, 2)}


if MAX_NEW_RUNS != 0 and not (os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')):
    raise RuntimeError('SURF_API_KEY and SURF_API_BASE are required for new M³ runs')

new_runs = 0
for item in tqdm(manifest['datasets'], desc='M³ fast pangaea descriptive'):
    path = m3_result_path(item['id'])
    if valid_m3_success(path, item):
        continue
    if 0 <= MAX_NEW_RUNS <= new_runs:
        continue
    print(f'Starting M³ pangaea {item["id"]}', flush=True)
    record = run_m3_one(item)
    write_json_atomic(path, record)
    new_runs += 1
    print(item['id'], record['status'], record.get('error', ''), 'trace:', record.get('trace_path'))
print('New M³ attempts:', new_runs)
print('saved successes:',
      sum(valid_m3_success(m3_result_path(item['id']), item) for item in manifest['datasets']),
      '/', len(manifest['datasets']))


M³ fast pangaea descriptive:   0%|          | 0/75 [00:00<?, ?it/s]

Starting M³ pangaea 897882
2026-10-19 10:57:49,828 - INFO - root - PlanExecutor initialized with topology: fast
2026-10-19 10:57:49,829 - INFO - root -   Players per step: 2
2026-10-19 10:57:49,829 - INFO - root -   Debate rounds: 1
2026-10-19 10:57:49,829 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-10-19 10:57:49,829 - INFO - root - Orchestrator initialized with topology: fast
2026-10-19 10:57:49,830 - INFO - root - Run trace: /home/com3dian/Github/metadata_agent/outputs/traces/897882_8d90a5d5f85742b48749d47ca5be40aa.jsonl
2026-10-19 10:57:49,831 - INFO - root - ============================================================
2026-10-19 10:57:49,831 - INFO - root - STARTING ORCHESTRATION
2026-10-19 10:57:49,831 - INFO - root - Context: 897882
2026-10-19 10:57:49,831 - INFO - root - Type: single_csv
2026-10-19 10:57:49,831 - INFO - root - Resources: ['